# 04 — SFLLD Leakage-Safe Missing-Value Treatment

**Input:** `SFLLD_modeling_dataset_complete_regular_and_strict_BLS_state_fallback.parquet`

**Final output:** `data/output/final_modeling_dataset/SFLLD_modeling_dataset_missing_values_treated.parquet`

This notebook implements the frozen missing-value strategy.

**Rules:** learned imputation statistics come from `Train` only; Validation/OOT/Other are transform-only; DTI gets a pre-imputation indicator; genuine categorical missingness becomes `"Missing"`; raw/intermediate macro lineage is not imputed; final resolved macro residuals get indicator + Train median; `Property_Valuation_Method` remains unimputed/excluded; all source columns are preserved.

The notebook performs missing-value treatment only. It does **not** perform scaling, one-hot encoding, rare-category grouping, or additional Feature Engineering.


In [1]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

INPUT_NAME = "SFLLD_modeling_dataset_complete_regular_and_strict_BLS_state_fallback.parquet"
OUTPUT_NAME = "SFLLD_modeling_dataset_missing_values_treated.parquet"

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for p in [start, *start.parents]:
        f = p / "data" / "output" / "final_modeling_dataset" / INPUT_NAME
        if f.exists():
            return p
    raise FileNotFoundError(
        f"Could not locate {INPUT_NAME}. Run from the SFLLD project root or notebooks/."
    )

ROOT = find_project_root()
INPUT = ROOT / "data" / "output" / "final_modeling_dataset" / INPUT_NAME
OUTPUT_DIR = ROOT / "data" / "output" / "final_modeling_dataset"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT = OUTPUT_DIR / OUTPUT_NAME
QA_DIR = ROOT / "data" / "output" / "quality_assurance" / "04_missing_value_treatment"
QA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", ROOT.resolve())
print("Input        :", INPUT.resolve())
print("Output       :", OUTPUT.resolve())
print("QA directory :", QA_DIR.resolve())


Project root : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6
Input        : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_modeling_dataset_complete_regular_and_strict_BLS_state_fallback.parquet
Output       : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_modeling_dataset_missing_values_treated.parquet
QA directory : C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\quality_assurance\04_missing_value_treatment


In [2]:
df_raw = pd.read_parquet(INPUT)
df = df_raw.copy(deep=True)

required = {"Loan_ID", "Split", "Target_24M"}
assert not (required - set(df.columns)), f"Missing core columns: {sorted(required-set(df.columns))}"
assert df["Loan_ID"].notna().all()
assert df["Loan_ID"].is_unique

split_norm = df["Split"].astype("string").str.strip().str.casefold()
train_mask = split_norm.eq("train")
assert train_mask.any(), "No Split == Train rows found."
assert (~train_mask).any(), "No holdout rows found."

print("Shape:", df.shape)
display(df["Split"].value_counts(dropna=False).rename_axis("Split").reset_index(name="rows"))
print(f"Train rows: {train_mask.sum():,}")


Shape: (713000, 105)


,Split,rows
0,Other Period,404240
1,Train,160003
2,OOT Test,99719
3,Validation,49038


Train rows: 160,003


## 1. Explicit treatment lists

No blanket dtype-based imputation is permitted.

In [3]:
LOAN_TRAIN_MEDIAN = [
    "FICO_Score", "MI_Percentage", "Number_of_Units",
    "Original_CLTV", "Original_LTV",
]
DTI_COL = "Original_DTI"
DTI_INDICATOR = "DTI_Missing"

CATEGORICAL_MISSING_LEVEL = ["First_Time_Homebuyer", "MSA"]
HIGH_MISSING_ABLATION = ["Special_Eligibility_Program"]
TEMPORAL_SUPPORT_EXCLUDE = ["Property_Valuation_Method"]

STRICT_FINAL_HOUSING = ["LS_Housing_HPI", "LS_Housing_HPI_Growth_12M"]
STRICT_FINAL_BLS = [
    "LS_BLS_Unemployment_Rate_Final",
    "LS_BLS_Employment_Growth_12M_Final",
    "LS_BLS_Unemployment_Rate_Change_12M_Final",
]

# Regular-PIT final-resolved benchmark variables.
REGULAR_FINAL_HOUSING = ["Housing_HPI", "Housing_HPI_Growth_12M"]
REGULAR_FINAL_BLS = [
    "BLS_Unemployment_Rate_Final",
    "BLS_Employment_Growth_12M_Final",
    "BLS_Unemployment_Rate_Change_12M_Final",
]

required_strategy = (
    LOAN_TRAIN_MEDIAN + [DTI_COL] + CATEGORICAL_MISSING_LEVEL
    + HIGH_MISSING_ABLATION + TEMPORAL_SUPPORT_EXCLUDE
    + STRICT_FINAL_HOUSING + STRICT_FINAL_BLS
)
missing_required = [c for c in required_strategy if c not in df.columns]
assert not missing_required, f"Required strategy columns absent: {missing_required}"

def validate_optional_group(name, cols):
    present = [c for c in cols if c in df.columns]
    if present and len(present) != len(cols):
        raise AssertionError(f"{name} partially present. Present={present}; missing={sorted(set(cols)-set(present))}")
    return bool(present)

HAS_REG_HOUSING = validate_optional_group("REGULAR_FINAL_HOUSING", REGULAR_FINAL_HOUSING)
HAS_REG_BLS = validate_optional_group("REGULAR_FINAL_BLS", REGULAR_FINAL_BLS)

print("Regular final housing detected:", HAS_REG_HOUSING)
print("Regular final BLS detected    :", HAS_REG_BLS)


Regular final housing detected: True
Regular final BLS detected    : True


## 2. Pre-treatment audit

In [4]:
def missing_summary(frame, cols):
    out = []
    for c in cols:
        if c in frame.columns:
            out.append({
                "column": c,
                "dtype": str(frame[c].dtype),
                "missing_all": int(frame[c].isna().sum()),
                "missing_all_pct": float(frame[c].isna().mean()*100),
                "missing_train": int(frame.loc[train_mask, c].isna().sum()),
                "missing_train_pct": float(frame.loc[train_mask, c].isna().mean()*100),
                "unique_non_null": int(frame[c].nunique(dropna=True)),
            })
    return pd.DataFrame(out)

strategy_cols = (
    LOAN_TRAIN_MEDIAN + [DTI_COL] + CATEGORICAL_MISSING_LEVEL
    + HIGH_MISSING_ABLATION + TEMPORAL_SUPPORT_EXCLUDE
    + STRICT_FINAL_HOUSING + STRICT_FINAL_BLS
    + (REGULAR_FINAL_HOUSING if HAS_REG_HOUSING else [])
    + (REGULAR_FINAL_BLS if HAS_REG_BLS else [])
)
pre_missing = missing_summary(df, strategy_cols)
display(pre_missing)
pre_missing.to_csv(QA_DIR / "04_pre_treatment_missingness.csv", index=False)


,column,dtype,missing_all,missing_all_pct,missing_train,missing_train_pct,unique_non_null
0,FICO_Score,float64,100,0.014025,15,0.009375,379
1,MI_Percentage,float64,3,0.000421,0,0.000000,28
2,Number_of_Units,float64,62,0.008696,0,0.000000,4
3,Original_CLTV,float64,11,0.001543,3,0.001875,314
4,Original_LTV,float64,10,0.001403,3,0.001875,297
5,Original_DTI,float64,78026,10.943338,16001,10.000437,63
6,First_Time_Homebuyer,string,13,0.001823,13,0.008125,2
7,MSA,string,89555,12.560309,18851,11.781654,455
8,Special_Eligibility_Program,str,683485,95.860449,158446,99.026893,3
9,Property_Valuation_Method,string,487320,68.347826,160003,100.000000,3


## 3. Final-macro missing-mask audit

A single shared residual indicator is used only when the group's missing masks are exactly identical.

In [5]:
def mask_audit(frame, cols, group):
    ref = frame[cols[0]].isna()
    rows = []
    all_same = True
    for c in cols:
        same = ref.equals(frame[c].isna())
        all_same &= same
        rows.append({"group": group, "column": c, "missing_count": int(frame[c].isna().sum()),
                     "same_mask_as_first": bool(same)})
    return pd.DataFrame(rows), bool(all_same)

tables = []
t, strict_housing_same = mask_audit(df, STRICT_FINAL_HOUSING, "Strict final housing"); tables.append(t)
t, strict_bls_same = mask_audit(df, STRICT_FINAL_BLS, "Strict final BLS"); tables.append(t)

reg_housing_same = None
reg_bls_same = None
if HAS_REG_HOUSING:
    t, reg_housing_same = mask_audit(df, REGULAR_FINAL_HOUSING, "Regular final housing"); tables.append(t)
if HAS_REG_BLS:
    t, reg_bls_same = mask_audit(df, REGULAR_FINAL_BLS, "Regular final BLS"); tables.append(t)

mask_audit_table = pd.concat(tables, ignore_index=True)
display(mask_audit_table)
mask_audit_table.to_csv(QA_DIR / "04_final_macro_missing_mask_audit.csv", index=False)


,group,column,missing_count,same_mask_as_first
0,Strict final housing,LS_Housing_HPI,329,True
1,Strict final housing,LS_Housing_HPI_Growth_12M,329,True
2,Strict final BLS,LS_BLS_Unemployment_Rate_Final,333,True
3,Strict final BLS,LS_BLS_Employment_Growth_12M_Final,333,True
4,Strict final BLS,LS_BLS_Unemployment_Rate_Change_12M_Final,333,True
5,Regular final housing,Housing_HPI,329,True
6,Regular final housing,Housing_HPI_Growth_12M,329,True
7,Regular final BLS,BLS_Unemployment_Rate_Final,412,True
8,Regular final BLS,BLS_Employment_Growth_12M_Final,412,True
9,Regular final BLS,BLS_Unemployment_Rate_Change_12M_Final,412,True


## 4. Fit Train-only medians

In [6]:
median_cols = (
    LOAN_TRAIN_MEDIAN + [DTI_COL] + STRICT_FINAL_HOUSING + STRICT_FINAL_BLS
    + (REGULAR_FINAL_HOUSING if HAS_REG_HOUSING else [])
    + (REGULAR_FINAL_BLS if HAS_REG_BLS else [])
)

train_medians = {}
for c in median_cols:
    med = pd.to_numeric(df.loc[train_mask, c], errors="coerce").median()
    assert pd.notna(med), f"Undefined Train median for {c}"
    train_medians[c] = float(med)

median_table = pd.DataFrame([{"column": c, "train_median": v} for c,v in train_medians.items()])
display(median_table)
median_table.to_csv(QA_DIR / "04_train_only_imputation_medians.csv", index=False)
(QA_DIR / "04_train_only_imputation_medians.json").write_text(
    json.dumps(train_medians, indent=2), encoding="utf-8"
)


,column,train_median
0,FICO_Score,757.000000
1,MI_Percentage,0.000000
2,Number_of_Units,1.000000
3,Original_CLTV,80.000000
4,Original_LTV,79.000000
5,Original_DTI,35.000000
6,LS_Housing_HPI,193.720000
7,LS_Housing_HPI_Growth_12M,0.052081
8,LS_BLS_Unemployment_Rate_Final,5.200000
9,LS_BLS_Employment_Growth_12M_Final,0.019572


633

## 5. Apply missing-value treatment

In [7]:
# T1
for c in LOAN_TRAIN_MEDIAN:
    df[c] = df[c].fillna(train_medians[c])

# T2: create DTI indicator BEFORE filling
df[DTI_INDICATOR] = df[DTI_COL].isna().astype("int8")
df[DTI_COL] = df[DTI_COL].fillna(train_medians[DTI_COL])

# T3
for c in CATEGORICAL_MISSING_LEVEL:
    df[c] = df[c].astype("string").fillna("Missing")

# T5: representation for optional ablation; not a baseline-feature decision
for c in HIGH_MISSING_ABLATION:
    df[c] = df[c].astype("string").fillna("Missing")

# Property_Valuation_Method intentionally remains unchanged.

def apply_final_macro_group(frame, cols, medians, shared_name, same_mask):
    created = []
    if same_mask:
        frame[shared_name] = frame[cols[0]].isna().astype("int8")
        created.append(shared_name)
    else:
        for c in cols:
            name = f"{c}_ResidualMissing"
            frame[name] = frame[c].isna().astype("int8")
            created.append(name)
    for c in cols:
        frame[c] = frame[c].fillna(medians[c])
    return created

created_indicators = [DTI_INDICATOR]
created_indicators += apply_final_macro_group(
    df, STRICT_FINAL_HOUSING, train_medians, "LS_Housing_ResidualMissing", strict_housing_same
)
created_indicators += apply_final_macro_group(
    df, STRICT_FINAL_BLS, train_medians, "LS_BLS_ResidualMissing", strict_bls_same
)
if HAS_REG_HOUSING:
    created_indicators += apply_final_macro_group(
        df, REGULAR_FINAL_HOUSING, train_medians, "Housing_Regular_ResidualMissing", reg_housing_same
    )
if HAS_REG_BLS:
    created_indicators += apply_final_macro_group(
        df, REGULAR_FINAL_BLS, train_medians, "BLS_Regular_ResidualMissing", reg_bls_same
    )

print("Created indicators:")
print(*created_indicators, sep="\n - ")


Created indicators:
DTI_Missing
 - LS_Housing_ResidualMissing
 - LS_BLS_ResidualMissing
 - Housing_Regular_ResidualMissing
 - BLS_Regular_ResidualMissing


## 6. Hard QA gates

The full source intentionally retains missing values in excluded/raw/intermediate lineage columns. Therefore this notebook does **not** require the entire DataFrame to become null-free. It requires every explicitly treated model field to be complete and every untouched source field to remain unchanged.


In [8]:
assert len(df) == len(df_raw), "Row count changed."
assert df["Loan_ID"].equals(df_raw["Loan_ID"]), "Loan_ID changed."
assert df["Split"].equals(df_raw["Split"]), "Split changed."
assert df["Target_24M"].equals(df_raw["Target_24M"]), "Target changed."
assert df["Loan_ID"].is_unique

new_cols = [c for c in df.columns if c not in df_raw.columns]
assert set(new_cols) == set(created_indicators), f"Unexpected new columns: {new_cols}"

must_complete = (
    LOAN_TRAIN_MEDIAN + [DTI_COL] + CATEGORICAL_MISSING_LEVEL + HIGH_MISSING_ABLATION
    + STRICT_FINAL_HOUSING + STRICT_FINAL_BLS
    + (REGULAR_FINAL_HOUSING if HAS_REG_HOUSING else [])
    + (REGULAR_FINAL_BLS if HAS_REG_BLS else [])
    + created_indicators
)
rem = df[must_complete].isna().sum()
assert (rem == 0).all(), "Missing remains in treated fields:\n" + rem[rem.gt(0)].to_string()

assert np.array_equal(df[DTI_INDICATOR].to_numpy(),
                      df_raw[DTI_COL].isna().astype("int8").to_numpy())

def verify_indicator(raw, treated, cols, shared_name, same_mask):
    if same_mask:
        assert np.array_equal(treated[shared_name].to_numpy(),
                              raw[cols[0]].isna().astype("int8").to_numpy())
    else:
        for c in cols:
            name = f"{c}_ResidualMissing"
            assert np.array_equal(treated[name].to_numpy(),
                                  raw[c].isna().astype("int8").to_numpy())

verify_indicator(df_raw, df, STRICT_FINAL_HOUSING, "LS_Housing_ResidualMissing", strict_housing_same)
verify_indicator(df_raw, df, STRICT_FINAL_BLS, "LS_BLS_ResidualMissing", strict_bls_same)
if HAS_REG_HOUSING:
    verify_indicator(df_raw, df, REGULAR_FINAL_HOUSING, "Housing_Regular_ResidualMissing", reg_housing_same)
if HAS_REG_BLS:
    verify_indicator(df_raw, df, REGULAR_FINAL_BLS, "BLS_Regular_ResidualMissing", reg_bls_same)

for c in CATEGORICAL_MISSING_LEVEL + HIGH_MISSING_ABLATION:
    m = df_raw[c].isna()
    if m.any():
        assert df.loc[m, c].eq("Missing").all(), f"{c}: explicit Missing conversion failed."

for c in TEMPORAL_SUPPORT_EXCLUDE:
    assert df[c].equals(df_raw[c]), f"{c} should remain untouched."

modified_source = set(
    LOAN_TRAIN_MEDIAN + [DTI_COL] + CATEGORICAL_MISSING_LEVEL + HIGH_MISSING_ABLATION
    + STRICT_FINAL_HOUSING + STRICT_FINAL_BLS
    + (REGULAR_FINAL_HOUSING if HAS_REG_HOUSING else [])
    + (REGULAR_FINAL_BLS if HAS_REG_BLS else [])
)
untouched = [c for c in df_raw.columns if c not in modified_source]
unexpected = [c for c in untouched if not df[c].equals(df_raw[c])]
assert not unexpected, f"Unexpectedly modified source columns: {unexpected}"

print("PASS — all treatment QA gates succeeded.")
print("Original shape:", df_raw.shape)
print("Treated shape :", df.shape)


PASS — all treatment QA gates succeeded.
Original shape: (713000, 105)
Treated shape : (713000, 110)


## 7. Post-treatment audit

In [9]:
post_missing = missing_summary(df, strategy_cols).rename(columns={
    "missing_all":"post_missing_all", "missing_all_pct":"post_missing_all_pct",
    "missing_train":"post_missing_train", "missing_train_pct":"post_missing_train_pct"
})
comparison = pre_missing.merge(
    post_missing[["column","post_missing_all","post_missing_all_pct","post_missing_train","post_missing_train_pct"]],
    on="column", how="left"
)
display(comparison)
comparison.to_csv(QA_DIR / "04_pre_vs_post_missingness.csv", index=False)

remaining_all = (
    df.isna().sum().rename("missing_count").to_frame()
      .assign(missing_pct=lambda x: x["missing_count"]/len(df)*100)
      .query("missing_count > 0").sort_values("missing_count", ascending=False)
)
print("Intentionally remaining/unprocessed missing columns:", len(remaining_all))
display(remaining_all)
remaining_all.to_csv(QA_DIR / "04_remaining_missingness_after_treatment.csv")


,column,dtype,missing_all,missing_all_pct,missing_train,missing_train_pct,unique_non_null,post_missing_all,post_missing_all_pct,post_missing_train,post_missing_train_pct
0,FICO_Score,float64,100,0.014025,15,0.009375,379,0,0.000000,0,0.0
1,MI_Percentage,float64,3,0.000421,0,0.000000,28,0,0.000000,0,0.0
2,Number_of_Units,float64,62,0.008696,0,0.000000,4,0,0.000000,0,0.0
3,Original_CLTV,float64,11,0.001543,3,0.001875,314,0,0.000000,0,0.0
4,Original_LTV,float64,10,0.001403,3,0.001875,297,0,0.000000,0,0.0
5,Original_DTI,float64,78026,10.943338,16001,10.000437,63,0,0.000000,0,0.0
6,First_Time_Homebuyer,string,13,0.001823,13,0.008125,2,0,0.000000,0,0.0
7,MSA,string,89555,12.560309,18851,11.781654,455,0,0.000000,0,0.0
8,Special_Eligibility_Program,str,683485,95.860449,158446,99.026893,3,0,0.000000,0,0.0
9,Property_Valuation_Method,string,487320,68.347826,160003,100.000000,3,487320,68.347826,160003,100.0


Intentionally remaining/unprocessed missing columns: 44


,missing_count,missing_pct
Property_Valuation_Method,487320,68.347826
FHFA_MSA_HPI_Growth_12M,148130,20.775596
FHFA_MSA_HPI,148130,20.775596
LS_FHFA_MSA_HPI_Growth_12M,148130,20.775596
LS_FHFA_MSA_HPI,148130,20.775596
LS_BLS_Unemployment_Rate_Change_12M,128686,18.048527
LS_BLS_Employment_Growth_12M,128686,18.048527
LS_BLS_Labor_Force,128686,18.048527
LS_BLS_Employment,128686,18.048527
LS_BLS_Unemployment_Rate,128686,18.048527


## 8. Save final treated Parquet and manifest

In [10]:
def loan_id_digest(s):
    h = hashlib.sha256()
    for v in s.astype("string").fillna("<NA>"):
        h.update(v.encode("utf-8")); h.update(b"\n")
    return h.hexdigest()

source_digest = loan_id_digest(df_raw["Loan_ID"])
assert loan_id_digest(df["Loan_ID"]) == source_digest

# Save only after every QA gate passes.
df.to_parquet(OUTPUT, index=False)

manifest = {
    "source_artifact": INPUT_NAME,
    "output_artifact": OUTPUT_NAME,
    "source_shape": [int(df_raw.shape[0]), int(df_raw.shape[1])],
    "output_shape": [int(df.shape[0]), int(df.shape[1])],
    "train_rows": int(train_mask.sum()),
    "split_counts": {str(k): int(v) for k,v in df["Split"].value_counts(dropna=False).items()},
    "fit_policy": "All learned imputation medians fitted on Train only; all other splits transform-only.",
    "train_medians": train_medians,
    "created_indicators": created_indicators,
    "strict_housing_identical_missing_mask": strict_housing_same,
    "strict_bls_identical_missing_mask": strict_bls_same,
    "regular_housing_identical_missing_mask": reg_housing_same,
    "regular_bls_identical_missing_mask": reg_bls_same,
    "property_valuation_method": "UNCHANGED / NO IMPUTATION / EXCLUDE PRIMARY",
    "special_eligibility_program": "Missing -> explicit Missing; OPTIONAL ABLATION, not baseline",
    "intentional_remaining_missing_columns": remaining_all.index.tolist(),
    "loan_id_order_sha256": source_digest,
}
manifest_path = QA_DIR / "04_missing_value_treatment_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

print("SAVED:")
print(" -", OUTPUT.resolve())
print(" -", manifest_path.resolve())


SAVED:
 - C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_modeling_dataset_missing_values_treated.parquet
 - C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\quality_assurance\04_missing_value_treatment\04_missing_value_treatment_manifest.json


## 9. Reload verification

In [11]:
saved = pd.read_parquet(OUTPUT)
assert saved.shape == df.shape
assert saved["Loan_ID"].equals(df["Loan_ID"])
assert saved["Split"].equals(df["Split"])
assert saved["Target_24M"].equals(df["Target_24M"])
assert loan_id_digest(saved["Loan_ID"]) == source_digest
assert saved[must_complete].isna().sum().sum() == 0

print("PASS — saved Parquet reloaded and verified.")
print("Final output:", OUTPUT.resolve())
print("Final shape :", saved.shape)


PASS — saved Parquet reloaded and verified.
Final output: C:\Users\itwill\Desktop\study\코딩공부\프로젝트\The Single Family Loan-Level Dataset (SFLLD) - 6\data\output\final_modeling_dataset\SFLLD_modeling_dataset_missing_values_treated.parquet
Final shape : (713000, 110)
